# Project: IT Support Chatbot

## Project Overview

**Problem**: TechCorp's IT team receives 30+ daily support tickets, with 70% being routine issues like password resets and VPN problems. This creates delays and prevents the IT team from focusing on strategic work.

**Solution**: Built an AI-powered chatbot using LangChain that can handle routine IT support questions instantly, escalate complex issues appropriately, and provide 24/7 basic support.

**Learning Goals**: 
- Master LangChain core components (Prompt Templates, LLMs, Chains, Memory)
- Build a practical business application
- Demonstrate real-world problem solving with AI

## Setup and Dependencies



In [1]:
# Import required libraries (updated for LangChain 0.3+)
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from typing import Dict

# Initialize local Ollama model (no API key needed!)
# Make sure you have Ollama installed and llama3.2 pulled:
# 1. Install: https://ollama.ai/download
# 2. Pull model: ollama pull llama3.2
# 3. Start: ollama serve

print("🤖 TechCorp IT Support Chatbot")
print("===============================")
print("Building intelligent IT support with LangChain + Local Ollama")
print("✅ No API keys required - runs completely local!")
print("✅ Using modern LangChain 0.3+ components")

🤖 TechCorp IT Support Chatbot
Building intelligent IT support with LangChain + Local Ollama
✅ No API keys required - runs completely local!
✅ Using modern LangChain 0.3+ components


## Step 1: Design Company-Specific Prompt Template

The key to a useful chatbot is a well-designed prompt that gives it personality, context, and clear guidelines about what it can and cannot handle.

In [2]:
# Create a prompt template with company context and clear capabilities
support_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template="""
    You are TechBot, TechCorp's friendly IT assistant.
    
    COMPANY CONTEXT:
    - Email System: outlook.techcorp.com
    - VPN: Connect via TechCorp VPN app
    - Employee IDs: Start with "TC-"
    - Help Portal: help.techcorp.com
    
    WHAT YOU CAN HELP WITH:
    ✅ Password resets and account access
    ✅ Email setup and troubleshooting  
    ✅ VPN connection issues
    ✅ Basic software installation
    ✅ Printer and file sharing problems
    
    WHAT TO ESCALATE TO HUMANS:
    ❌ Hardware failures or repairs
    ❌ Security breaches or suspicious activity
    ❌ Server outages or network failures
    ❌ Advanced configuration changes
    
    CONVERSATION HISTORY: {history}
    CURRENT QUESTION: {input}
    
    Provide helpful, step-by-step guidance. If you cannot help, say "I need to escalate this to our IT team" and explain why.
    
    Response:
    """
)

print("✅ Prompt template created with TechCorp-specific context")

✅ Prompt template created with TechCorp-specific context


## Step 2: Build Memory-Enabled Chatbot Class

Now we'll create a chatbot class that combines our prompt template with conversation memory and basic performance tracking.

In [3]:
class ITSupportBot:
    """
    IT Support Chatbot that demonstrates:
    - Prompt Templates (company-specific responses)
    - RunnableWithMessageHistory (modern memory across interactions)
    - LLM Integration (Ollama for local intelligent responses)
    """

    def __init__(self):
        # Initialize local Ollama model (no API key needed!)
        self.llm = OllamaLLM(
            model="llama3.2",  # or "mistral", "codellama"
            temperature=0.3
        )

        # Create runnable chain with memory
        self.chain = support_prompt | self.llm

        # Store chat histories by session
        self.store: Dict[str, InMemoryChatMessageHistory] = {}

        # Create memory-enabled conversation chain
        self.conversation = RunnableWithMessageHistory(
            self.chain,
            self.get_session_history,
            input_messages_key="input",
            history_messages_key="history"
        )

        # Track basic performance metrics
        self.stats = {
            "issues_resolved": 0,
            "escalations": 0,
            "total_conversations": 0
        }

    def get_session_history(self, session_id: str) -> BaseChatMessageHistory:
        """Get or create chat history for a session"""
        if session_id not in self.store:
            self.store[session_id] = InMemoryChatMessageHistory()
        return self.store[session_id]

    def _should_escalate(self, user_question: str, response_text: str) -> bool:
        """Decide escalation using flexible response cues + critical issue keywords."""
        response_lower = response_text.lower()
        question_lower = user_question.lower()

        escalation_phrases = [
            "escalate",
            "escalated",
            "cannot help",
            "can't help",
            "human support",
            "it team",
            "support team",
            "open a ticket",
            "create a ticket"
        ]

        critical_issue_keywords = [
            "server is down",
            "entire company server",
            "network outage",
            "security breach",
            "ransomware",
            "data center",
            "nobody can work"
        ]

        response_signals = any(phrase in response_lower for phrase in escalation_phrases)
        critical_issue = any(keyword in question_lower for keyword in critical_issue_keywords)

        return response_signals or critical_issue

    def help(self, user_question, session_id="default_session"):
        """Main method to handle user questions"""
        self.stats["total_conversations"] += 1

        # Get response from the conversation chain with memory
        response = self.conversation.invoke(
            {"input": user_question},
            config={"configurable": {"session_id": session_id}}
        )

        # Normalize to plain text for reliable matching and display
        response_text = response if isinstance(response, str) else str(response)

        # Robust escalation detection (not dependent on one exact phrase)
        if self._should_escalate(user_question, response_text):
            self.stats["escalations"] += 1
            response_text += "\n\n🎫 Ticket #TC-{} created for human review.".format(
                1000 + self.stats["escalations"]
            )
        else:
            self.stats["issues_resolved"] += 1

        return response_text

    def get_performance_stats(self):
        """Return chatbot performance metrics"""
        total = self.stats["total_conversations"]
        if total == 0:
            return "No conversations yet"

        resolution_rate = (self.stats["issues_resolved"] / total) * 100
        return {
            "Total Conversations": total,
            "Issues Resolved by AI": self.stats["issues_resolved"],
            "Escalated to Humans": self.stats["escalations"],
            "AI Resolution Rate": f"{resolution_rate:.1f}%"
        }

# Initialize the chatbot (no API key needed!)
chatbot = ITSupportBot()
print("✅ IT Support chatbot initialized with modern LangChain + local Ollama + memory")

✅ IT Support chatbot initialized with modern LangChain + local Ollama + memory


## Step 3: Test Core Functionality

Let's test our chatbot with realistic scenarios to see how it handles different types of requests.

In [4]:
print("🧪 TESTING CORE FUNCTIONALITY")
print("=" * 35)

# Test 1: Common issue that chatbot should handle
print("\n1️⃣ TEST: Password Reset (Should Handle)")
print("Employee: I forgot my password and can't access my email")

response1 = chatbot.help("I forgot my password and can't access my email")
print(f"TechBot: {response1}")

🧪 TESTING CORE FUNCTIONALITY

1️⃣ TEST: Password Reset (Should Handle)
Employee: I forgot my password and can't access my email
TechBot: Don't worry, I'm here to help! Forgetting your password is a common issue, and we can easily resolve it. Here are the steps to reset your password:

**Step 1: Go to the Outlook login page**
Open a web browser (e.g., Google Chrome, Mozilla Firefox) and navigate to outlook.techcorp.com.

**Step 2: Click on "Forgot Password"**
On the login page, click on the "Forgot Password" link below the username field.

**Step 3: Enter your Employee ID**
Enter your full Employee ID (starting with "TC-") in the required field. If you're not sure what your Employee ID is, check your email or ask your manager.

**Step 4: Answer security questions (if applicable)**
If you have previously set up security questions, answer them correctly to proceed.

**Step 5: Create a new password**
Choose a strong and unique password for your Outlook account. Make sure it meets our compa

In [5]:
# Test 2: Memory capabilities
print("\n2️⃣ TEST: Memory Functionality")
print("Employee: My employee ID is TC-12345")

response2 = chatbot.help("My employee ID is TC-12345")
print(f"TechBot: {response2}")

print("\nEmployee: What was my employee ID again?")
response3 = chatbot.help("What was my employee ID again?")
print(f"TechBot: {response3}")


2️⃣ TEST: Memory Functionality
Employee: My employee ID is TC-12345
TechBot: You've provided your Employee ID as TC-12345. Now that we have the necessary information, let's move forward with resetting your password.

To confirm, I'll repeat the steps for you:

1. Go to the Outlook login page: Open a web browser (e.g., Google Chrome, Mozilla Firefox) and navigate to outlook.techcorp.com.
2. Click on "Forgot Password": On the login page, click on the "Forgot Password" link below the username field.
3. Enter your Employee ID: Enter your full Employee ID (TC-12345) in the required field. If you're not sure what your Employee ID is, check your email or ask your manager.
4. Answer security questions (if applicable): If you have previously set up security questions, answer them correctly to proceed.
5. Create a new password: Choose a strong and unique password for your Outlook account. Make sure it meets our company's password policy requirements.
6. Verify your identity (optional): You may 

In [6]:
# Test 3: Complex issue that should be escalated
print("\n3️⃣ TEST: Escalation Logic (Should Escalate)")
print("Employee: The entire company server is down and nobody can work")

response4 = chatbot.help("The entire company server is down and nobody can work")
print(f"TechBot: {response4}")


3️⃣ TEST: Escalation Logic (Should Escalate)
Employee: The entire company server is down and nobody can work
TechBot: I'm TechBot, and I'm here to help you with your issue. However, I've been informed that the entire company server is currently down, which means we're experiencing a network failure.

Unfortunately, as a friendly IT assistant, I don't have the authority to resolve this type of critical issue. Network failures typically require human intervention from our IT team to diagnose and fix.

I need to escalate this to our IT team to ensure that everyone's work is not disrupted. They will be able to investigate the cause of the failure, perform any necessary repairs, and restore the server to a functional state as soon as possible.

In the meantime, I recommend checking the company's help portal (help.techcorp.com) for any updates on the status of the server outage or follow our social media channels for announcements from our IT team. We'll keep you informed about any progress

## Step 4: Performance Analysis

How well the chatbot performed and what business impact it could have.

In [7]:
# Display chatbot performance metrics
print("\n📊 CHATBOT PERFORMANCE")
print("=" * 25)

performance = chatbot.get_performance_stats()
for metric, value in performance.items():
    print(f"{metric}: {value}")


📊 CHATBOT PERFORMANCE
Total Conversations: 4
Issues Resolved by AI: 3
Escalated to Humans: 1
AI Resolution Rate: 75.0%


## Step 5: Interactive Demo

Try the chatbot yourself with your own questions! This section lets you test the chatbot interactively.

In [ ]:
# Interactive Demo - Notebook Friendly Version
def test_multiple_scenarios():
    """Test chatbot with predefined scenarios - works great in notebooks!"""
    print("\n🎮 INTERACTIVE DEMO - Multiple Test Scenarios")
    print("=" * 45)
    
    test_cases = [
        "I can't connect to the VPN",
        "My employee ID is TC-98765",
        "How do I install Microsoft Teams?",
        "What was my employee ID again?",                   # Tests memory
        "The entire data center is on fire!",               # Should escalate
        "I need access to the shared marketing folder"
    ]
    
    for i, question in enumerate(test_cases, 1):
        print(f"\n{i}️⃣ Employee: {question}")
        try:
            response = chatbot.help(question)
            print(f"🤖 TechBot: {response}")
        except Exception as e:
            print(f"❌ Error: {e}")
        
        # Show stats after a few interactions
        if i == 3:
            print("\n📊 Midway Performance:")
            stats = chatbot.get_performance_stats()
            for metric, value in stats.items():
                print(f"   {metric}: {value}")
            print("-" * 30)
    
    # Final performance summary
    print("\n📈 FINAL PERFORMANCE SUMMARY")
    print("=" * 30)
    final_stats = chatbot.get_performance_stats()
    for metric, value in final_stats.items():
        print(f"{metric}: {value}")

# Run the demo
test_multiple_scenarios()

# Optional: Single question testing
print("\n💡 Want to test a specific question? Run this:")
print("response = chatbot.help('Your question here')")
print("print(response)")

## Project Summary & Reflection

Successfully created an IT support chatbot that demonstrates all core LangChain concepts:

- **✅ Prompt Templates**: Company-specific response formatting with clear capabilities and limitations
- **✅ LLM Integration**: llama3.2 for intelligent, context-aware responses  
- **✅ Conversation Memory**: Remembers context across multiple interactions
- **✅ Chain Composition**: Combines prompts, memory, and LLM into a cohesive workflow

## Future Enhancements
### Technical Improvements
- Integrate with the company ticketing system (auto-create/update incidents)
- Connect to internal knowledge base for policy and troubleshooting answers
- Add sentiment analysis to detect frustrated users and escalate faster
- Implement feedback collection after each interaction for continuous improvement

### Deployment Priorities
- Deploy to Slack/Microsoft Teams for employee self-service support
- Add a web chat widget for the company intranet/help portal

### Advanced Capabilities
- Build an analytics dashboard for IT managers (volume, resolution, satisfaction)
- Learn from resolved tickets to improve future response quality
- Add multi-language support for global teams
- Integrate with HR systems for employee verification and role-based support access